In [1]:
#TEIC-module1.0 (expert model)


filepath1 ="./TEIC_data(expert)(TEIC-table2).csv" #filepath to the dataset (TEIC-table3)
filepath2 ="." #filepath to the folder where to save the training dataset (expert data)
filepath3 ="." #filepath to the folder where to save the testing dataset (expert data)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import learning_curve

    
data = pd.read_csv(filepath1)
ID = data[["Index"]]
parameter_list = ["Age","Body weight","BMI","Creatinine clearance","Alb","T1","T2","T3","T4","T5","T6","T7","TO",
                  "Bacteremia","Bone infection","Cholecystitis","Intra-abdominal infection","Pneumonia","Sepsis","Skin infection","Unknown","Urinary tract infection","Vascular graft Infection",
                  "MRSA","CNS (MR)","Enterococcus faecium","Corynebacterium",
                  "ACEI/ARB","Loop diuretic","Diuretic","NSAIDs","ST","Aciclovir/Valaciclovir","Tacrolimus"]
parameter = data[parameter_list]
parameter["TO"].replace("AM", 0, inplace=True)
parameter["TO"].replace("PM", 1, inplace=True)
display(parameter)
loading = data[["Loading dose"]]
maintenance = data[["Maintenance dose"]] 
TDM = data[["TDM","TDM exclusion criteria"]]

#random state and train_test_split
rs = 1 
ID_train, ID_test, parameter_train, parameter_test, loading_train, loading_test, maintenance_train, maintenance_test, TDM_train, TDM_test = train_test_split(ID,parameter, loading, maintenance,TDM, test_size=0.2,random_state=rs)
train = pd.concat([ID_train,parameter_train,loading_train,maintenance_train,TDM_train],axis=1).set_index("Index")
test = pd.concat([ID_test,parameter_test,loading_test,maintenance_test,TDM_test],axis=1).set_index("Index")
display(train)
display(test)
train.to_csv(str(filepath2)+"/TEIC_non-ICU_train_rs{}(TEIC-table9).csv".format(rs), index=True)    
test.to_csv(str(filepath3)+"/TEIC_non-ICU_test_rs{}(TEIC-table10).csv".format(rs), index=True)      

#GridSearchCV
grid_rfc = {"n_estimators":[10,20,40,80,160,320],"max_depth":[2,4,8,16,32,64]}
grid_search_loading_rfc = GridSearchCV(RandomForestClassifier(random_state=rs), grid_rfc,cv=5)
grid_search_maintenance_rfc = GridSearchCV(RandomForestClassifier(random_state=rs), grid_rfc,cv=5)

grid_search_loading_rfc.fit(parameter_train, loading_train)
grid_search_maintenance_rfc.fit(parameter_train, maintenance_train)

#training phase
rfc_loading_train_score = grid_search_loading_rfc.score(parameter_train,loading_train) #score
rfc_maintenance_train_score = grid_search_maintenance_rfc.score(parameter_train,maintenance_train) #score

#testing phase
rfc_loading_test_score = grid_search_loading_rfc.score(parameter_test,loading_test) #score
rfc_maintenance_test_score = grid_search_maintenance_rfc.score(parameter_test,maintenance_test) #score
rfc_loading_params = grid_search_loading_rfc.best_params_ #best parameter
rfc_maintenance_params = grid_search_maintenance_rfc.best_params_ #best parameter

#show feature importance
print("feature importance for loading dose is {}".format(grid_search_loading_rfc.best_estimator_.feature_importances_))
print("feature importance for maintenance dose is {}".format(grid_search_maintenance_rfc.best_estimator_.feature_importances_))


/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/pandas/core/generic.py:6619: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return self._update_inplace(result)


,Age,Body weight,BMI,Creatinine clearance,Alb,T1,T2,T3,T4,T5,...,CNS (MR),Enterococcus faecium,Corynebacterium,ACEI/ARB,Loop diuretic,Diuretic,NSAIDs,ST,Aciclovir/Valaciclovir,Tacrolimus
0,79,44.40,20.519068,43.208333,2.8,0,0,1,0,0,...,0,1,0,1,1,1,0,0,0,0
1,99,48.70,24.357356,21.625828,3.1,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,86,56.00,21.207101,23.595506,3.8,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,41,65.50,23.429809,117.774039,3.1,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,1
4,51,55.70,21.408511,20.180580,1.6,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,54,85.30,30.622198,103.965419,3.3,0,1,0,0,0,...,1,0,0,0,1,0,0,0,0,0
114,44,78.05,31.068077,90.261905,3.5,0,1,0,0,0,...,0,0,0,1,0,1,1,0,0,0
115,84,54.20,20.399714,45.821256,2.2,1,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
116,81,42.90,19.532637,34.745397,2.4,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


,Age,Body weight,BMI,Creatinine clearance,Alb,T1,T2,T3,T4,T5,...,Loop diuretic,Diuretic,NSAIDs,ST,Aciclovir/Valaciclovir,Tacrolimus,Loading dose,Maintenance dose,TDM,TDM exclusion criteria
Index,,,,,,,,,,,,,,,,,,,,,
70,77,54.4,22.098002,58.765432,3.7,0,0,0,1,0,...,0,0,1,0,0,0,400mg 4 doses,400 mg,None,0
96,71,46.0,17.901556,51.862745,2.5,0,1,0,0,0,...,0,0,0,0,0,0,400mg 4 doses,400 mg,None,0
36,74,63.2,23.073904,43.233831,1.5,0,1,0,0,0,...,1,0,1,0,0,0,400mg 4 doses,400 mg,21.6,0
45,69,60.0,21.107467,55.817610,2.3,0,0,0,0,0,...,0,0,0,0,0,0,600mg 4 doses,400 mg,25,0
86,88,56.4,20.716253,110.090090,1.3,0,0,1,0,0,...,0,0,0,0,0,0,400mg 3 doses,0 mg,18.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10,70,51.3,21.772114,49.295058,2.5,0,0,0,1,0,...,0,0,1,0,0,0,500mg 4 doses,300 mg,25.9,0
73,66,57.8,22.105880,56.042977,2.0,1,0,0,0,0,...,0,0,0,0,0,0,400mg 4 doses,400 mg,None,0
13,81,75.5,29.826795,23.704236,2.5,0,1,0,0,0,...,0,0,1,0,0,0,600mg 3 doses,300 mg,18.9,0


,Age,Body weight,BMI,Creatinine clearance,Alb,T1,T2,T3,T4,T5,...,Loop diuretic,Diuretic,NSAIDs,ST,Aciclovir/Valaciclovir,Tacrolimus,Loading dose,Maintenance dose,TDM,TDM exclusion criteria
Index,,,,,,,,,,,,,,,,,,,,,
95,35,77.80,24.444992,166.850490,1.9,0,0,0,1,0,...,0,0,1,0,0,0,800mg 4 doses,600 mg,26.3,0
55,71,59.15,27.004200,26.186198,1.2,0,0,1,0,0,...,1,0,0,1,0,0,400mg 4 doses,0 mg,25.4,2
60,84,41.90,16.084478,47.924837,2.4,0,0,1,0,0,...,0,0,0,0,0,0,400mg 3 doses,0 mg,21,0
116,84,54.20,20.399714,45.821256,2.2,1,0,0,0,0,...,0,0,0,0,0,0,400mg 4 doses,300 mg,20.7,0
75,86,47.30,18.899407,27.080153,2.3,0,0,1,0,0,...,1,0,0,0,0,0,400mg 3 doses,0 mg,18.4,0
47,63,63.60,22.006920,64.777778,2.7,0,0,1,0,0,...,0,0,1,0,0,0,600mg 3 doses,0 mg,18,0
32,61,66.80,26.423005,103.231612,2.7,0,0,0,1,0,...,1,0,0,1,1,0,600mg 4 doses,0 mg,40,0
81,76,46.00,19.048282,80.826873,2.3,0,1,0,0,0,...,0,0,1,0,0,0,400mg 4 doses,400 mg,None,0
49,72,56.30,20.958031,42.200176,2.8,0,0,0,0,1,...,0,0,0,0,0,0,400mg 4 doses,300 mg,16,0


/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_split.py:666: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(("The least populated class in y has only %d"
/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:598: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:598: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:598: DataCo

feature importance for loading dose is [0.11032048 0.16355188 0.12271883 0.12128268 0.07946497 0.02677042
 0.01781714 0.02719304 0.012598   0.01032086 0.         0.0033759
 0.04215561 0.01922471 0.01012132 0.0122092  0.0136208  0.0069044
 0.00301896 0.01592628 0.01182393 0.010632   0.00279053 0.02337969
 0.01457891 0.02400326 0.00798534 0.01517707 0.02072971 0.01659832
 0.01540578 0.00875924 0.00217327 0.00736746]
feature importance for maintenance dose is [0.09166535 0.09051298 0.09243913 0.16092647 0.10513526 0.01875053
 0.02535682 0.10018744 0.02900828 0.01043517 0.         0.00165577
 0.02264891 0.02887464 0.01172368 0.00696761 0.00702882 0.00650155
 0.00088218 0.01962435 0.00835758 0.00440976 0.00304178 0.01984207
 0.01355549 0.01477953 0.0063508  0.00793009 0.02437448 0.02464609
 0.02661281 0.00982062 0.0030019  0.00295205]


/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_search.py:880: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self.best_estimator_.fit(X, y, **fit_params)


In [ ]:
#TEIC-module2.0 (expert model)


filepath1 ="./TEIC_data(expert)(TEIC-table2).csv" #filepath to the dataset (TEIC-table3)
filepath2 ="." #filepath to the folder where to save the training dataset (expert data)
filepath3 ="." #filepath to the folder where to save the testing dataset (expert data)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import learning_curve

    
data = pd.read_csv(filepath1)
ID = data[["Index"]]
parameter_list = ["Age","Body weight","BMI","Creatinine clearance","Alb","T3"]
parameter = data[parameter_list]

display(parameter)
loading = data[["Loading dose"]]g
maintenance = data[["Maintenance dose"]] 
TDM = data[["TDM","TDM exclusion criteria"]]


#random state and train_test_split
rs = 1 
ID_train, ID_test, parameter_train, parameter_test, loading_train, loading_test, maintenance_train, maintenance_test, TDM_train, TDM_test = train_test_split(ID,parameter, loading, maintenance,TDM, test_size=0.2,random_state=rs)
train = pd.concat([ID_train,parameter_train,loading_train,maintenance_train,TDM_train],axis=1).set_index("Index")
test = pd.concat([ID_test,parameter_test,loading_test,maintenance_test,TDM_test],axis=1).set_index("Index")
display(train)
display(test)
train.to_csv(str(filepath2)+"/TEIC_non-ICU_train_rs{}(TEIC-table11).csv".format(rs), index=True)    
test.to_csv(str(filepath3)+"/TEIC_non-ICU_test_rs{}(TEIC-table12).csv".format(rs), index=True)      


#GridSearchCV
grid_rfc = {"n_estimators":[10,20,40,80,160,320],"max_depth":[2,4,8,16,32,64]}
grid_search_loading_rfc = GridSearchCV(RandomForestClassifier(random_state=rs), grid_rfc,cv=5)
grid_search_maintenance_rfc = GridSearchCV(RandomForestClassifier(random_state=rs), grid_rfc,cv=5)

grid_search_loading_rfc.fit(parameter_train, loading_train)
grid_search_maintenance_rfc.fit(parameter_train, maintenance_train)

#training phase
rfc_loading_train_score = grid_search_loading_rfc.score(parameter_train,loading_train) #score
rfc_maintenance_train_score = grid_search_maintenance_rfc.score(parameter_train,maintenance_train) #score

#testing phase
rfc_loading_test_score = grid_search_loading_rfc.score(parameter_test,loading_test) #score
rfc_maintenance_test_score = grid_search_maintenance_rfc.score(parameter_test,maintenance_test) #score
rfc_loading_params = grid_search_loading_rfc.best_params_ #best parameter
rfc_maintenance_params = grid_search_maintenance_rfc.best_params_ #best parameter

#show best parameters and scores
print("train score for loading dose is {}".format(rfc_loading_train_score))
print("train score for maiantenance dose is {}".format(rfc_maintenance_train_score))

print("test score for loading dose is {}".format(rfc_loading_test_score))
print("test parameters for loading dose is {}".format(rfc_loading_params))

print("test score for maiantenance dose is {}".format(rfc_maintenance_test_score))
print("test parameters for maintenance dose is {}".format(rfc_maintenance_params))

test["loading dose (ML)"] = grid_search_loading_rfc.predict(parameter_test)
test["maintenance dose (ML)"] = grid_search_maintenance_rfc.predict(parameter_test)
display(test)
test.to_csv(str(filepath3)+"/TEIC_result_(non-ICU_expertML)_internal-validation_rs{}(TEIC-table13).csv".format(rs), index=True)  

#show feature importance
print("feature importance for loading dose is {}".format(grid_search_loading_rfc.best_estimator_.feature_importances_))

,Age,Body weight,BMI,Creatinine clearance,Alb,T3
0,79,44.40,20.519068,43.208333,2.8,1
1,99,48.70,24.357356,21.625828,3.1,0
2,86,56.00,21.207101,23.595506,3.8,0
3,41,65.50,23.429809,117.774039,3.1,0
4,51,55.70,21.408511,20.180580,1.6,1
...,...,...,...,...,...,...
113,54,85.30,30.622198,103.965419,3.3,0
114,44,78.05,31.068077,90.261905,3.5,0
115,84,54.20,20.399714,45.821256,2.2,0
116,81,42.90,19.532637,34.745397,2.4,0


,Age,Body weight,BMI,Creatinine clearance,Alb,T3,Loading dose,Maintenance dose,TDM,TDM exclusion criteria
Index,,,,,,,,,,
70,77,54.4,22.098002,58.765432,3.7,0,400mg 4 doses,400 mg,None,0
96,71,46.0,17.901556,51.862745,2.5,0,400mg 4 doses,400 mg,None,0
36,74,63.2,23.073904,43.233831,1.5,0,400mg 4 doses,400 mg,21.6,0
45,69,60.0,21.107467,55.817610,2.3,0,600mg 4 doses,400 mg,25,0
86,88,56.4,20.716253,110.090090,1.3,1,400mg 3 doses,0 mg,18.8,0
...,...,...,...,...,...,...,...,...,...,...
10,70,51.3,21.772114,49.295058,2.5,0,500mg 4 doses,300 mg,25.9,0
73,66,57.8,22.105880,56.042977,2.0,0,400mg 4 doses,400 mg,None,0
13,81,75.5,29.826795,23.704236,2.5,0,600mg 3 doses,300 mg,18.9,0


,Age,Body weight,BMI,Creatinine clearance,Alb,T3,Loading dose,Maintenance dose,TDM,TDM exclusion criteria
Index,,,,,,,,,,
95,35,77.80,24.444992,166.850490,1.9,0,800mg 4 doses,600 mg,26.3,0
55,71,59.15,27.004200,26.186198,1.2,1,400mg 4 doses,0 mg,25.4,2
60,84,41.90,16.084478,47.924837,2.4,1,400mg 3 doses,0 mg,21,0
116,84,54.20,20.399714,45.821256,2.2,0,400mg 4 doses,300 mg,20.7,0
75,86,47.30,18.899407,27.080153,2.3,1,400mg 3 doses,0 mg,18.4,0
47,63,63.60,22.006920,64.777778,2.7,1,600mg 3 doses,0 mg,18,0
32,61,66.80,26.423005,103.231612,2.7,0,600mg 4 doses,0 mg,40,0
81,76,46.00,19.048282,80.826873,2.3,0,400mg 4 doses,400 mg,None,0
49,72,56.30,20.958031,42.200176,2.8,0,400mg 4 doses,300 mg,16,0


/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_split.py:666: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(("The least populated class in y has only %d"
/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:598: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:598: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:598: DataCo

train score for loading dose is 1.0
train score for maiantenance dose is 0.776595744680851
test score for loading dose is 0.4583333333333333
test parameters for loading dose is {'max_depth': 16, 'n_estimators': 80}
test score for maiantenance dose is 0.6666666666666666
test parameters for maintenance dose is {'max_depth': 4, 'n_estimators': 160}


/Users/tetsuomatsuzaki/opt/anaconda3/lib/python3.9/site-packages/sklearn/model_selection/_search.py:880: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self.best_estimator_.fit(X, y, **fit_params)


,Age,Body weight,BMI,Creatinine clearance,Alb,T3,Loading dose,Maintenance dose,TDM,TDM exclusion criteria,loading dose (ML),maintenance dose (ML)
Index,,,,,,,,,,,,
95,35,77.80,24.444992,166.850490,1.9,0,800mg 4 doses,600 mg,26.3,0,600mg 4 doses,400 mg
55,71,59.15,27.004200,26.186198,1.2,1,400mg 4 doses,0 mg,25.4,2,600mg 3 doses,0 mg
60,84,41.90,16.084478,47.924837,2.4,1,400mg 3 doses,0 mg,21,0,400mg 3 doses,0 mg
116,84,54.20,20.399714,45.821256,2.2,0,400mg 4 doses,300 mg,20.7,0,400mg 4 doses,300 mg
75,86,47.30,18.899407,27.080153,2.3,1,400mg 3 doses,0 mg,18.4,0,400mg 3 doses,0 mg
47,63,63.60,22.006920,64.777778,2.7,1,600mg 3 doses,0 mg,18,0,600mg 4 doses,0 mg
32,61,66.80,26.423005,103.231612,2.7,0,600mg 4 doses,0 mg,40,0,600mg 4 doses,400 mg
81,76,46.00,19.048282,80.826873,2.3,0,400mg 4 doses,400 mg,None,0,400mg 4 doses,400 mg
49,72,56.30,20.958031,42.200176,2.8,0,400mg 4 doses,300 mg,16,0,400mg 4 doses,400 mg


feature importance for loading dose is [0.15910705 0.29214188 0.20879354 0.16666093 0.13489738 0.03839923]


In [ ]:
#TEIC-module4.0 (learning curve)


filepath1 ="./TEIC_data(expert)(TEIC-table2).csv" #filepath to the dataset (TEIC-table3)
filepath2 ="." #filepath to the folder where to save the training dataset (expert data)
filepath3 ="." #filepath to the folder where to save the testing dataset (expert data)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import learning_curve

    
data = pd.read_csv(filepath1)
ID = data[["Index"]]
parameter_list = ["Age","Body weight","BMI","Creatinine clearance","Alb","T3"]
parameter = data[parameter_list]
display(parameter)
loading = data[["Loading dose"]]
maintenance = data[["Maintenance dose"]] 
TDM = data[["TDM","TDM exclusion criteria"]]


#random state and train_test_split
rs = 1 
ID_train, ID_test, parameter_train, parameter_test, loading_train, loading_test, maintenance_train, maintenance_test, TDM_train, TDM_test = train_test_split(ID,parameter, loading, maintenance,TDM, test_size=0.2,random_state=rs)
train = pd.concat([ID_train,parameter_train,loading_train,maintenance_train,TDM_train],axis=1).set_index("Index")
test = pd.concat([ID_test,parameter_test,loading_test,maintenance_test,TDM_test],axis=1).set_index("Index")
display(train)
display(test)
train.to_csv(str(filepath2)+"/TEIC_non-ICU_train_rs{}(TEIC-table9).csv".format(rs), index=True)    
test.to_csv(str(filepath3)+"/TEIC_non-ICU_test_rs{}(TEIC-table10).csv".format(rs), index=True)      


#Learning curve
grid_rfc = {"n_estimators":[10,20,40,80,160,320],"max_depth":[2,4,8,16,32,64]}
grid_search_loading_rfc = GridSearchCV(RandomForestClassifier(random_state=rs), grid_rfc,cv=5)
grid_search_maintenance_rfc = GridSearchCV(RandomForestClassifier(random_state=rs), grid_rfc,cv=5)

    #loading
train_size_abs, train_scores, test_scores = learning_curve(
    grid_search_loading_rfc, parameter, loading, train_sizes=[0.3, 0.5, 0.75,1.0])

for train_size, cv_train_scores, cv_test_scores in zip(train_size_abs, train_scores, test_scores):
    print(f"{train_size} samples were used to train the model(loading)")
    print(f"The average train accuracy (loading doses) is {cv_train_scores.mean():.2f}")
    print(f"The average test accuracy (loading doses) is {cv_test_scores.mean():.2f}")

    #maintenance
train_size_abs, train_scores, test_scores = learning_curve(
    grid_search_loading_rfc, parameter, maintenance, train_sizes=[0.3, 0.5, 0.75,1.0])

for train_size, cv_train_scores, cv_test_scores in zip(train_size_abs, train_scores, test_scores):
    print(f"{train_size} samples were used to train the model(maintenance)")
    print(f"The average train accuracy (maintenance doses) is {cv_train_scores.mean():.2f}")
    print(f"The average test accuracy (maintenance doses) is {cv_test_scores.mean():.2f}")

,Age,Body weight,BMI,Creatinine clearance,Alb,T3
0,79,44.40,20.519068,43.208333,2.8,1
1,99,48.70,24.357356,21.625828,3.1,0
2,86,56.00,21.207101,23.595506,3.8,0
3,41,65.50,23.429809,117.774039,3.1,0
4,51,55.70,21.408511,20.180580,1.6,1
...,...,...,...,...,...,...
113,54,85.30,30.622198,103.965419,3.3,0
114,44,78.05,31.068077,90.261905,3.5,0
115,84,54.20,20.399714,45.821256,2.2,0
116,81,42.90,19.532637,34.745397,2.4,0


,Age,Body weight,BMI,Creatinine clearance,Alb,T3,Loading dose,Maintenance dose,TDM,TDM exclusion criteria
Index,,,,,,,,,,
70,77,54.4,22.098002,58.765432,3.7,0,400mg 4 doses,400 mg,None,0
96,71,46.0,17.901556,51.862745,2.5,0,400mg 4 doses,400 mg,None,0
36,74,63.2,23.073904,43.233831,1.5,0,400mg 4 doses,400 mg,21.6,0
45,69,60.0,21.107467,55.817610,2.3,0,600mg 4 doses,400 mg,25,0
86,88,56.4,20.716253,110.090090,1.3,1,400mg 3 doses,0 mg,18.8,0
...,...,...,...,...,...,...,...,...,...,...
10,70,51.3,21.772114,49.295058,2.5,0,500mg 4 doses,300 mg,25.9,0
73,66,57.8,22.105880,56.042977,2.0,0,400mg 4 doses,400 mg,None,0
13,81,75.5,29.826795,23.704236,2.5,0,600mg 3 doses,300 mg,18.9,0


,Age,Body weight,BMI,Creatinine clearance,Alb,T3,Loading dose,Maintenance dose,TDM,TDM exclusion criteria
Index,,,,,,,,,,
95,35,77.80,24.444992,166.850490,1.9,0,800mg 4 doses,600 mg,26.3,0
55,71,59.15,27.004200,26.186198,1.2,1,400mg 4 doses,0 mg,25.4,2
60,84,41.90,16.084478,47.924837,2.4,1,400mg 3 doses,0 mg,21,0
116,84,54.20,20.399714,45.821256,2.2,0,400mg 4 doses,300 mg,20.7,0
75,86,47.30,18.899407,27.080153,2.3,1,400mg 3 doses,0 mg,18.4,0
47,63,63.60,22.006920,64.777778,2.7,1,600mg 3 doses,0 mg,18,0
32,61,66.80,26.423005,103.231612,2.7,0,600mg 4 doses,0 mg,40,0
81,76,46.00,19.048282,80.826873,2.3,0,400mg 4 doses,400 mg,None,0
49,72,56.30,20.958031,42.200176,2.8,0,400mg 4 doses,300 mg,16,0


/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_split.py:676: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_split.py:676: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using r

28 samples were used to train the model(loading)
The average train accuracy (loading doses) is 0.81
The average test accuracy (loading doses) is 0.32
47 samples were used to train the model(loading)
The average train accuracy (loading doses) is 0.60
The average test accuracy (loading doses) is 0.40
70 samples were used to train the model(loading)
The average train accuracy (loading doses) is 0.76
The average test accuracy (loading doses) is 0.42
94 samples were used to train the model(loading)
The average train accuracy (loading doses) is 0.83
The average test accuracy (loading doses) is 0.44


/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages

28 samples were used to train the model(maintenance)
The average train accuracy (maintenance doses) is 0.96
The average test accuracy (maintenance doses) is 0.43
47 samples were used to train the model(maintenance)
The average train accuracy (maintenance doses) is 0.83
The average test accuracy (maintenance doses) is 0.57
70 samples were used to train the model(maintenance)
The average train accuracy (maintenance doses) is 0.93
The average test accuracy (maintenance doses) is 0.60
94 samples were used to train the model(maintenance)
The average train accuracy (maintenance doses) is 0.94
The average test accuracy (maintenance doses) is 0.54


/Users/matsuzakitetsuo/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_search.py:926: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  self.best_estimator_.fit(X, y, **fit_params)
